# Un Algoritmo di Correzione per un Motore di Ricerca

## Caso d'Uso Aziendale: Searchify

### Introduzione all'Azienda
Searchify è una startup innovativa che offre un motore di ricerca personalizzato per aziende di medie dimensioni, consentendo di cercare informazioni specifiche nei loro database interni. Uno dei principali vantaggi di Searchify è la semplicità d'uso: gli utenti possono trovare rapidamente documenti, report e altre risorse aziendali digitando parole chiave. Tuttavia, l'azienda si trova ad affrontare un problema che mina l'esperienza dell'utente.

### Problema
Il problema principale di Searchify è che molti utenti commettono errori di digitazione durante l'inserimento delle parole chiave. Questi errori causano risultati di ricerca nulli o non pertinenti, portando a insoddisfazione tra gli utenti. Ad esempio, se un utente cerca "raporto vendite 2023" invece di "rapporto vendite 2023", il motore di ricerca non restituisce alcun risultato.

### Obiettivo del Progetto
L'obiettivo è sviluppare un algoritmo di correzione automatica per il motore di ricerca di Searchify. Questo algoritmo dovrà:

1. Rilevare automaticamente gli errori di digitazione o le parole non valide.
2. Suggerire la parola corretta più probabile.
3. Restituire risultati pertinenti basati sulla correzione suggerita.

L'implementazione di questa funzionalità migliorerà notevolmente l'esperienza utente, aumentando l'efficienza del motore di ricerca e la soddisfazione dei clienti.

### Benefici Attesi
- Miglioramento dell'accuratezza: Riduzione degli errori di ricerca dovuti a digitazioni errate.
- Incremento della produttività: Gli utenti troveranno le informazioni più velocemente.
- Aumento della fedeltà dei clienti: Un'esperienza utente più fluida e soddisfacente porterà a una maggiore adozione del prodotto.

### Specifiche del Progetto
1. Input: Una stringa rappresentante una query di ricerca, digitata dall'utente.
2. Output:
    - Se la query contiene errori, il sistema suggerisce una correzione.
    - Se la query è corretta, restituisce la stessa query.
3. Funzionalità Chiave:
    - Confronto tra la parola inserita e un dizionario di parole corrette (da predefinire nel codice).
    - Implementazione di un algoritmo che calcoli la "distanza di edit" (es. distanza di Levenshtein) per trovare le parole più simili.
    - Gestione di casi d'uso realistici come errori di battitura comuni, lettere scambiate, omissioni o aggiunte.

### Consegna
Scrivi un programma Python che implementi l'algoritmo di correzione automatica per il motore di ricerca. Il tuo codice deve:

1. Contenere una funzione principale chiamata suggest_correction(query, dictionary).
    - Parametri:
        - query: La stringa di input inserita dall'utente.
        - dictionary: Una lista di parole corrette.
    - Ritorno:
        - La parola corretta più probabile o la query originale se è già valida.
2. Testare il funzionamento con almeno 10 casi d'uso (inclusi errori comuni e query corrette).
3. Essere ben documentato, con commenti che spieghino le principali scelte implementative.
4. Fornire un dizionario base contenente almeno 50 parole.

### Nota
Il progetto deve essere completato senza l'uso di librerie esterne per il calcolo della distanza di edit o per altre funzionalità. L'obiettivo è che lo studente implementi l'algoritmo interamente da zero.

## Soluzione Proposta
Questa funzione implementa l'algoritmo della distanza di Levenshtein. La distanza di Levenshtein, o distanza di edit, è una misura per la differenza fra due stringhe. Essa serve a determinare quanto due stringhe siano simili.

La distanza di Levenshtein tra due stringhe A e B è il numero minimo di modifiche elementari che consentono di trasformare la A nella B. Per modifica elementare si intende

- la cancellazione di un carattere
- la sostituzione di un carattere con un altro
- l'inserimento di un carattere

![levensthein_distance](levensthein_distance.webp)

La matrice `distances_matrix` rappresenta i sotto problemi di trasformazione. la cella `distances_matrix[i][j]` contiene la distanza di Levenshtein tra i prefissi primi i caratteri di word_1 e primi j caratteri di word_2. Per poter rappresentare anche il caso in cui uno dei prefissi sia la stringa vuota serve una riga e una colonna aggiuntive che vengono inizializzate in questo modo:

- Prima riga: `distances_matrix[0][j]` é la distanza tra la stringa vuota e il prefisso di word_2 di lunghezza j -> servono j inserimenti quindi `distances_matrix[0][j]=j`.
- Prima colonna `distances_matrix[i][0]` è distanza tra il prefisso di word_1 di lunghezza i e la stringa vuota -> servono i cancellazioni quindi `distances_matrix[i][0]=i`.

In [ ]:
def levenshtein_distance(word_1, word_2):
    """
    Calcola la distanza di Levenshtein tra due parole.
    La distanza di Levenshtein è il numero minimo di operazioni (inserimento, cancellazione, sostituzione) necessarie per trasformare una parola nell'altra.
    
    Args:
        word_1 (str): Prima parola
        word_2 (str): Seconda parola
        
    Returns:
        int: La distanza di Levenshtein tra le due parole
    """

    len_1 = len(word_1)
    len_2 = len(word_2)

    # Crea una matrice (len1+1) x (len2+1) inizializzando tutti i valori a 0
    distances_matrix = [[0] * (len_2 + 1) for _ in range(len_1 + 1)]

    # Inizializza la prima riga e colonna
    for i in range(len_1 + 1):
        distances_matrix[i][0] = i
    for j in range(len_2 + 1):
        distances_matrix[0][j] = j

    # Riempie la matrice
    for i in range(1, len_1 + 1):
        for j in range(1, len_2 + 1):
            if word_1[i - 1] == word_2[j - 1]:
                distances_matrix[i][j] = distances_matrix[i - 1][j - 1]
            else:
                # Prende il minimo tra:
                # - Sostituzione (diagonale): matrix[i-1][j-1] + 1
                # - Cancellazione (sopra): matrix[i-1][j] + 1
                # - Inserimento (sinistra): matrix[i][j-1] + 1
                distances_matrix[i][j] = 1 + min(distances_matrix[i - 1][j - 1],
                                                 distances_matrix[i - 1][j],
                                                 distances_matrix[i][j - 1])

    return distances_matrix[len_1][len_2]


def suggest_correction(query, dictionary, lowercase=False):
    """
    Suggerisce una correzione per una parola digitata in modo errato.
    Confronta la query con tutte le parole nel dizionario e suggerisce quella con la minore distanza di Levenshtein.
    
    Args:
        query (str): La parola digitata dall'utente
        dictionary (list): Lista di parole corrette
        lowercase (bool): Se True, ignora le differenze tra maiuscole e minuscole
        
    Returns:
        str: La parola corretta suggerita o la query originale se è già corretta
        int: La distanza di Levenshtein tra la query e la parola suggerita
    """

    if not isinstance(dictionary, list) or not dictionary:
        raise ValueError("Il dizionario deve essere una lista non vuota.")

    if not isinstance(query, str):
        raise ValueError("La query deve essere una stringa.")

    if not query or not query.strip():
        raise ValueError("La query non può essere vuota.")

    distances = []

    if lowercase:
        # Se la query è già nel dizionario, ritorna la query originale in minuscolo
        if query.lower() in [word.lower() for word in dictionary]:
            return query.lower(), 0

        # Calcola la distanza per ogni parola nel dizionario
        for word in dictionary:
            distance = levenshtein_distance(query.lower(), word.lower())
            distances.append((word, distance))

    else:
        # Se la query è già nel dizionario, ritorna la query originale
        if query in dictionary:
            return query, 0

        # Calcola la distanza per ogni parola nel dizionario
        for word in dictionary:
            distance = levenshtein_distance(query, word)
            distances.append((word, distance))

    # Ordina per distanza in quanto la parola più simile sarà la prima
    distances.sort(key=lambda x: x[1])

    suggested_word = distances[0][0]
    min_distance = distances[0][1]

    return suggested_word, min_distance

## Database e casi d'uso per i test

In [ ]:
database = [
    "rapporto", "vendite", "cliente", "prodotto", "servizio", "azienda",
    "database", "software", "hardware", "rete", "server", "documento",
    "progetto", "budget", "fattura", "ricevuta", "contratto", "accordo",
    "strategia", "marketing", "pubblicità", "analisi", "dati", "statistiche",
    "ricerca", "risultato", "errore", "correzione", "motore", "query",
    "parola", "lettera", "digitazione", "modello", "algoritmo", "funzione",
    "variabile", "lista", "dizionario", "stringa", "numero", "intero",
    "profilo", "account", "password", "utente", "nome", "cognome",
    "indirizzo", "città", "paese", "telefono", "email", "messaggio",
    "aggiorna", "crea", "modifica", "elimina", "salva", "carica",
    "esporta", "importa", "scarica", "condividi", "invia", "ricevi"
]

tests = [
    # (query, risultato atteso)
    ("raporto", "rapporto (omissione di lettera)"),
    ("clinte", "cliente (omissione di lettera)"),
    ("databse", "database (omissione di lettera)"),
    ("sofware", "software (omissione di lettera)"),
    ("uteeente", "utente (inserimento di lettera)"),
    ("algoritmop", "algoritmo (inserimento di lettera)"),
    ("analissi", "analisi (inserimento di lettera)"),
    ("rendite", "vendite (errore di digitazione)"),
    ("podello", "modello (errore di digitazione)"),
    ("agienda", "azienda (errore di digitazione)"),
    ("prodotto", "prodotto (parola già corretta)"),
    ("servizio", "servizio (parola già corretta)"),
    ("MESsaggio", "messaggio (errore di digitazione con lettere maiuscole)"),
    ("", "Exception: query vuota (ValueError)"),
    ("  ", "Exception: query con soli spazi (ValueError)")
]

tests_lowercase = [
    # (query, risultato atteso)
    ("RAPPORTO", "rapporto (parola già corretta)"),
    ("SoFwArE", " software (omissione di lettera)"),
    ("ALGoRitNp", "algoritmo (errore di digitazione)"),
    ("aNaLiSSi", "analisi (inserimento di lettera)"),
]

## Test del codice

In [ ]:
def test_suggest_correction(dictionary, test_cases):
    """
    Testa la funzione suggest_correction con una serie di casi d'uso.

    Args:
        dictionary (list): Il dizionario di parole corrette
        test_cases (list): Una lista di tuple (query, descrizione) per i test
    """

    print("TEST DEL SISTEMA DI CORREZIONE AUTOMATICA")
    print()

    for i, (query, description) in enumerate(test_cases, 1):
        try:
            suggestion, distance = suggest_correction(query, dictionary)

            print(f"Test {i}: INPUT: '{query}'")
            print(f"  - SUGGERIMENTO: '{suggestion}'")
            print(f"  - DISTANZA DI EDIT: {distance}")
            print(f"  - DESCRIZIONE: {description}")
            print()

        except ValueError as e:
            print(f"Test {i}: INPUT: '{query}'")
            print(f"  - ERRORE: {e}")
            print(f"  - DESCRIZIONE: {description}")
            print()


def test_suggest_correction_lowercase(dictionary, test_cases_lowercase):
    """
    Testa la funzione suggest_correction con il parametro lowercase=True.

    Args:
        dictionary (list): Il dizionario di parole corrette
        test_cases_lowercase (list): Una lista di tuple (query, expected_suggestion, description) per i test case con lowercase=True
    """

    print("TEST CON PARAMETRO LOWERCASE=TRUE (CASE-INSENSITIVE)")
    print()

    for i, (query, description) in enumerate(test_cases_lowercase, 1):
        try:
            suggestion, distance = suggest_correction(query, dictionary, lowercase=True)

            print(f"Test {i}: INPUT: '{query}'")
            print(f"  - SUGGERIMENTO: '{suggestion}'")
            print(f"  - DISTANZA DI EDIT: {distance}")
            print(f"  - DESCRIZIONE: {description}")
            print()

        except ValueError as e:
            print(f"Test {i}: INPUT: '{query}'")
            print(f"  - ERRORE: {e}")
            print(f"  - DESCRIZIONE: {description}")
            print()

In [ ]:
test_suggest_correction(database, tests)
print()
print()
test_suggest_correction_lowercase(database, tests_lowercase)